# Aula 3, LSTM

Notebook executável que acompanha a aula [03-lstm.md](../../lessons/modulo-05-deep-learning-nlp/03-lstm.md).

## O que vamos fazer aqui

Em vez de treinar uma LSTM, vamos demonstrar diretamente por que ela funciona,
ajustando os portões à mão. O contraste com a RNN simples é gritante: a RNN esquece um
sinal em poucos passos, enquanto a LSTM segura a memória por dezenas de passos. Só
numpy.

## Demonstração 1, a RNN simples esquece

Damos um sinal de valor 1 e, depois, só entradas zero. Com peso recorrente contrativo,
o estado decai a cada passo, e a informação inicial desaparece.

In [ ]:
import numpy as np


def sigmoide(z):
    return 1 / (1 + np.exp(-z))


Wh = 0.5
print("RNN simples (entrada zero após o início):")
for t in [0, 5, 25]:
    valor = 1.0
    for _ in range(t):
        valor = np.tanh(Wh * valor)
    print(f"  passo {t:2d}: h = {valor:.4f}")

O estado cai de 1,0 para cerca de 0,03 em 5 passos e chega a praticamente 0 em
25. A memória da RNN simples é curta, ela dilui o passado a cada passo.

## Demonstração 2, a LSTM preserva

Agora colocamos um valor na memória e configuramos os portões para segurá-lo: o de
esquecimento perto de 1 e o de entrada perto de 0. Alimentamos com zeros e observamos a
célula.

In [ ]:
def lstm_passo(x, h, c, bf=6.0, bi=-6.0, bo=2.0):
    f = sigmoide(bf)                    # esquecimento perto de 1
    i = sigmoide(bi)                    # entrada perto de 0
    o = sigmoide(bo)                    # saída moderada
    g = np.tanh(x)
    c = f * c + i * g                   # memória quase intacta
    h = o * np.tanh(c)
    return h, c


c = 1.0; h = 0.0                        # memória inicial guardada na célula
print("LSTM (mesma situação, entrada zero):")
for passo in range(26):
    if passo in (0, 5, 25):
        print(f"  passo {passo:2d}: c = {c:.4f}")
    h, c = lstm_passo(0.0, h, c)

A memória da LSTM vai de 1,0 e ainda vale cerca de 0,94 após 25 passos. A
esteira de memória preservou a informação, graças à atualização aditiva do estado de
célula com o portão de esquecimento perto de 1. É por isso que a LSTM aprende
dependências longas que a RNN simples não alcança. Para o projeto, monte a curva de
retenção das duas arquiteturas lado a lado.